# Structured Outputs

**Prerequisites:** `01_your_first_llm.ipynb`, `04_prompting.ipynb`

Almost every agent needs to turn an LLM reply into a Python object it can act on — not just print. This notebook builds that pipeline: JSON mode, a Pydantic schema, validation, and a retry loop for when validation fails, because on this model it will.

**By the end you'll have:**
- Seen free-text output vary in structure across identical calls
- Used JSON mode, and confirmed exactly what it does and doesn't guarantee
- Triggered a real JSON parsing failure
- Validated a reply against a Pydantic schema, and seen it correctly reject bad data
- Built `extract()` in `llm.py` — JSON mode + validation + retry, wrapped up

In [1]:
import sys
import json
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from pydantic import BaseModel, ValidationError

from llm import chat, extract, DEFAULT_MODEL

## Why this matters

If you ask for something in plain English and just eyeball the reply, it looks fine. The problem shows up when *code*, not you, has to parse it — and the exact same prompt doesn't reliably produce the exact same structure twice:

In [2]:
for _ in range(3):
    reply = chat(
        [{"role": "user", "content": "Give me a recipe for pancakes with a title, ingredients, and total time."}],
        temperature=1,
    )
    print(repr(reply[:80]))

'**Fluffy Classic Pancakes**\n\n**Total Time:** 20\u202fminutes (Prep\u202f≈\u202f10\u202fmin\u202f+\u202fCook\u202f≈\u202f'


'**Fluffy Classic Pancakes**\n\n**Ingredients**  \n- 1\u202f½ cups (190\u202fg) all‑purpose fl'


'**Fluffy Classic Pancakes**\n\n**Ingredients**  \n- 1\u202f½ cups (190\u202fg) all‑purpose fl'


Headers, formatting, even whether "Ingredients" gets a `###` in front of it — all of that drifts run to run. Any regex or string-splitting logic you write against one of these will break on another. That's the actual problem structured output solves; it's not about making things "look nicer."

## JSON mode

`response_format={"type": "json_object"}` constrains the model to emit syntactically valid JSON — nothing more.

In [3]:
reply = chat(
    [{"role": "user", "content": (
        "Give me a recipe for pancakes as JSON with fields "
        "title, ingredients (list), minutes (int)."
    )}],
    response_format={"type": "json_object"},
)
print(reply)

{"title":"Classic Pancakes","ingredients":["1 1/2 cups all-purpose flour","3 1/2 teaspoons baking powder","1 teaspoon salt","1 tablespoon sugar","1 1/4 cups milk","1 egg","3 tablespoons melted butter"],"minutes":20}


Two things worth knowing, both from hitting them directly:

1. **The word "json" has to appear somewhere in your messages**, or Groq (like OpenAI) rejects the request outright with a 400 — `response_format` alone isn't enough.
2. **"Valid JSON" isn't "matches my schema."** JSON mode guarantees the syntax parses; it says nothing about which fields exist or what type they are. `llama-3.3-70b-versatile`, which this chapter was written against, doesn't support Groq's stricter `json_schema` mode — that is limited to the `openai/gpt-oss-*` models. So on that model syntax is guaranteed and structure isn't, which is exactly what the rest of this notebook deals with.

## Parsing failures

Drop `response_format` and just ask nicely — a real, common failure mode shows up:

In [4]:
unstructured_reply = chat([{"role": "user", "content": (
    "Give me a recipe for pancakes as JSON with fields title, ingredients (list), minutes (int)."
)}])
print(repr(unstructured_reply[:60]))

try:
    json.loads(unstructured_reply)
except json.JSONDecodeError as e:
    print("JSONDecodeError:", e)

'```json\n{\n  "title": "Classic Fluffy Pancakes",\n  "ingredien'
JSONDecodeError: Expecting value: line 1 column 1 (char 0)


The model wrapped the JSON in a ` ```json ` markdown fence — helpful in a chat UI, fatal to `json.loads()`. This is exactly why JSON mode exists: without it, even asking nicely for JSON doesn't reliably produce parseable output. JSON mode fixes this specific failure — but as noted above, it doesn't guarantee the parsed data matches your schema, which is what validation, next, is for.

## Pydantic models

A schema says precisely what "correct" means, and gives you a typed object back instead of a dict you're hoping has the right keys:

In [5]:
class Recipe(BaseModel):
    title: str
    ingredients: list[str]
    minutes: int

In [6]:
recipe = Recipe.model_validate_json(reply)
print(type(recipe))
print(recipe)
print(recipe.minutes, "minutes,", len(recipe.ingredients), "ingredients")

<class '__main__.Recipe'>
title='Classic Pancakes' ingredients=['1 1/2 cups all-purpose flour', '3 1/2 teaspoons baking powder', '1 teaspoon salt', '1 tablespoon sugar', '1 1/4 cups milk', '1 egg', '3 tablespoons melted butter'] minutes=20
20 minutes, 7 ingredients


That's the actual point: `recipe.minutes` is a real `int` you can do arithmetic on, `recipe.ingredients` is a real `list`, not text you'd have to re-parse.

## Validation catches real problems

JSON mode would happily accept this — it's syntactically valid JSON. It just doesn't match `Recipe`:

In [7]:
bad_json = '{"title": "Pancakes", "ingredients": "flour, milk", "minutes": "twenty"}'

try:
    Recipe.model_validate_json(bad_json)
except ValidationError as e:
    print(e)

2 validation errors for Recipe
ingredients
  Input should be a valid array [type=list_type, input_value='flour, milk', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type
minutes
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='twenty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


Two independent errors, each naming the exact field and what was wrong with it: `ingredients` was a string instead of a list, `minutes` was a string that isn't even a valid integer. This is what "validation" buys you over just calling `json.loads()` — a schema mismatch is caught immediately, with a precise, actionable error, instead of surfacing later as a confusing bug somewhere downstream.

## Retries

The idea: call, try to validate, and if it fails, tell the model *exactly* what was wrong and ask it to fix it, using the real error message as feedback. Walked through by hand — suppose attempt 1 comes back like this:

In [8]:
attempt_1 = '{"title": "Pancakes", "ingredients": "flour, milk, sugar", "minutes": 20}'

try:
    Recipe.model_validate_json(attempt_1)
except ValidationError as e:
    error = str(e)
    print("attempt 1 failed:")
    print(error)

attempt 1 failed:
1 validation error for Recipe
ingredients
  Input should be a valid array [type=list_type, input_value='flour, milk, sugar', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type


In [9]:
messages = [
    {"role": "user", "content": "Give me a recipe for pancakes as JSON."},
    {"role": "assistant", "content": attempt_1},
    {"role": "user", "content": f"That was invalid: {error}\nReturn corrected JSON only, matching the schema."},
]

attempt_2 = chat(messages, response_format={"type": "json_object"})
recipe = Recipe.model_validate_json(attempt_2)
print("attempt 2 succeeded:", recipe)

attempt 2 succeeded: title='Pancakes' ingredients=['flour', 'milk', 'sugar', 'egg', 'baking powder', 'salt', 'butter'] minutes=20


Feeding the real validation error back as a follow-up message is enough for the model to fix its own mistake. Looping that — call, validate, feed the error back on failure, stop after N attempts — is `extract()` in `llm.py`:

```python
def extract(prompt, schema, retries=3, model=DEFAULT_MODEL):
    messages = [{"role": "user", "content": prompt}]
    for _ in range(retries):
        reply = chat(messages, model=model, response_format={"type": "json_object"})
        messages.append({"role": "assistant", "content": reply})
        try:
            return schema.model_validate_json(reply)
        except (json.JSONDecodeError, ValidationError) as e:
            messages.append({
                "role": "user",
                "content": f"That was invalid: {e}\nReturn corrected JSON only, matching the schema.",
            })
    raise ValueError(f"Failed to get valid {schema.__name__} after {retries} attempts")
```

In [10]:
recipe = extract("Give me a recipe for pancakes, as JSON.", Recipe)
print(type(recipe))
print(recipe)

<class '__main__.Recipe'>
title='Classic Pancakes' ingredients=['1 cup all-purpose flour', '2 tablespoons sugar', '2 teaspoons baking powder', '1/2 teaspoon salt', '1 cup milk', '1 egg', '2 tablespoons melted butter'] minutes=20


### If your model *does* support `json_schema`

`DEFAULT_MODEL` is now `openai/gpt-oss-120b`, which does. Passing

```python
response_format={"type": "json_schema",
                 "json_schema": {"name": "person", "schema": Person.model_json_schema()}}
```

makes the API itself enforce the field names and types, and `extract()` now asks for that first, falling back to plain JSON mode on any model that rejects it.

**That does not make this notebook's technique redundant, and it is worth being precise about why.** A schema constrains *shape*. It cannot constrain *sense*.

Measured, from chapter 2's `SelectThink` — asked to score three candidates out of ten, and to return the indices of the best two:

```json
{"keep_indices": [20], "scores": [869.0]}
```

An index of 20 into a list of length three. One score for three candidates. A score of 869 out of a stated maximum of 10. **Every one of those is schema-valid** — `keep_indices` is a list of integers, `scores` is a list of numbers — and every one of them is nonsense.

No JSON schema can express "one entry per candidate", "an index that exists in the list above", or "between 0 and 10 as I asked". So validation after the fact stays, the caller still sanity-checks what it was handed, and the retry still feeds the real error back. Schema enforcement removes one class of failure at the source; it does not remove the need to look at what you got.

## What you built

✓ Watched free-text formatting actually vary across identical calls

✓ Used JSON mode and confirmed precisely what it does and doesn't guarantee

✓ Triggered a real JSON parsing failure from markdown-fenced output

✓ Defined `Recipe` and validated against it, and seen validation correctly reject bad data with field-level errors

✓ Built `extract()` in `llm.py` — JSON mode, validation, and retry-with-feedback in one call

**Next:** `06_function_calling.ipynb` — tool schemas and a single call→execute→result round trip.